In [7]:
import csv
import io
import unicodedata

import numpy as np


### 3. Cahier des charges
L'application devra gérer un ensemble de plantes. 
Chaque plante possède au minimum les informations suivantes : 
- nom vernaculaire ; 
-  nom scientifique ; 
-  famille botanique ; 
-  cycle ; 
-  besoins ; 
-  photo. 

### 4. Classe plante
méthode `__init__` et `__str__`

Ne sera pas utilisé par la suite. Inclu dans la fonction herbier

In [ ]:
class plante:
    def __init__(self, nom, nom_scientifique, famille, cycle, besoins, photo):
        self.nom = nom
        self.nom_scientifique = nom_scientifique
        self.famille = famille
        self.cycle = cycle
        self.besoins = besoins
        self.photo = photo

    def __str__(self):
        return f"{self.nom} ({self.nom_scientifique}) - {self.famille} Cycle : {self.cycle} Besoins : {self.besoins}"


## 5. Gestion de l'herbier
contenant les méthodes : 
- `ajouter_plante` :  Ajoute une plante à l'herbier en indiquant nom, nom_scientifique, famille, cycle, besoins, photo
- `supprimer_plante` : Supprime les plantes correspondant à un/des critères donnés
- `recherche` : Recherche des plantes selon des mots-clés | tolère $n$ `diff` avec les mots clefs de la recherche
- `nettoyer` : Nettoie et normalise une ligne ou une liste de lignes
- `export herbier` : Exporte/Sauvegarde les données de l'herbier dans un fichier CSV
- `import herbier` : Lit un fichier CSV et le transforme en tableau NumPy

In [ ]:
class herbier:
    PHOTO_VIDE = "aucune_photo.png"            # "null" des photos (utilisable plus tard comme image par défaut)
    VALEURS_VIDES = ("", "nan", "none", "null")

    # Seuls les noms de famille qui ne suivent pas la règle -aceae / -acées
    FAMILLES_EXCEPTIONS = {  # noqa: RUF012
        "compositae": "asteraceae",   "composees": "asteraceae",
        "cruciferae": "brassicaceae", "cruciferes": "brassicaceae",
        "leguminosae": "fabaceae",    "legumineuses": "fabaceae",
        "umbelliferae": "apiaceae",   "ombelliferes": "apiaceae",
        "labiatae": "lamiaceae",      "labiees": "lamiaceae",
        "gramineae": "poaceae",       "graminees": "poaceae",
        "renonculacees": "ranunculaceae", "renonculacee": "ranunculaceae",
    }

    @staticmethod
    def _sans_accent(texte):
        texte = texte.replace("œ", "oe").replace("æ", "ae")
        texte = unicodedata.normalize("NFD", texte)
        return "".join(c for c in texte if not unicodedata.combining(c))

    def _famille_latine(self, famille):
        '''famille : déjà en minuscules et sans accent. Renvoie le nom latin.'''
        if famille in self.FAMILLES_EXCEPTIONS:
            return self.FAMILLES_EXCEPTIONS[famille]
        for fin in ("acees", "acee"):            # solanacées / solanacée
            if famille.endswith(fin):
                return famille[:-len(fin)] + "aceae"
        return famille

    def __init__(self, fichier_herbier):
        '''
        Initialise l'herbier en chargeant les plantes depuis le fichier CSV.
        L'attribut self.list_plante contient toutes les plantes sous forme de tableau NumPy.
        '''
        self.fichier_herbier = fichier_herbier
        L = self.import_herbier(fichier_herbier)
        print(L)
        self.carac = L[0,:]
        self.list_plante = L[1:,:]

    def ajouter_plante(self, info_plante):
        '''
        Ajoute une plante à l'herbier.
        info_plante : list
            Liste de 6 éléments dans l'ordre :
            [nom, nom_scientifique, famille, cycle, besoins, photo]
        '''
        if isinstance(info_plante, list) and len(info_plante) == 6:
            info_plante = self.nettoyer(info_plante)          # [[...]]
            existantes = {tuple(l) for l in self.nettoyer(self.list_plante, silencieux=True)}
            if tuple(info_plante[0]) in existantes:
                print(f"[doublon] {info_plante[0]} existe déjà, non ajoutée")
                return
            self.export_herbier(self.fichier_herbier, info_plante, ecraser=False)
            self.list_plante = np.vstack((self.list_plante, info_plante))
        else:
            raise TypeError(f"la liste {info_plante} n'est pas au bon format [nom, nom_scientifique, famille, cycle, besoins, photo]")

    def supprimer_plante(self, att, nbr=0):
        '''
        Supprime les plantes correspondant à un critère donné.
        att : str ou list
            Terme ou liste de termes à rechercher.
        nbr : int, optionnel
            Nombre maximal de plantes à supprimer. Si nbr=0, on supprime toutes les correspondances.
        '''
        l_supp = self.rechercher(att, diff=0, supprimer=True)

        if l_supp == "aucune correspondance avec votre recherche":
            return "aucune correspondance avec votre recherche, rien à supprimer"

        match_supp = self.list_plante == l_supp[0]
        match_supp = match_supp.sum(axis=1)
        idx_supp = np.where(match_supp == match_supp.max())[0]

        if nbr == 0:
            n = len(idx_supp)
            self.list_plante = np.delete(self.list_plante, idx_supp, axis=0)
            return f"{n} lignes supprimées : {l_supp[:n]} supprimé"
        else:
            n = min(len(idx_supp), nbr)
            self.list_plante = np.delete(self.list_plante, idx_supp[-n:], axis=0)
            return f"{n} lignes supprimées : {l_supp[:n]} supprimé, il reste {len(idx_supp)-n} valide pour ce tag"

    def rechercher(self, rech, diff=0, supprimer=False):
        """
        Recherche des plantes selon des mots-clés.
        rech : str ou list
            Chaîne de caractères ou liste de mots-clés à rechercher.
        diff : int, optionnel
            Nombre d'écarts autorisés par rapport à la recherche.
        supprimer : bool, optionnel
            Si True, renvoie directement les plantes correspondantes sous forme de liste.
        """
        if isinstance(rech, str):
            mots_recherche = rech.lower().replace("'", " ").split()
        else:
            mots_recherche = []
            for element in rech:
                mots_recherche.extend(
                    str(element).lower().replace("'", " ").split()
                )

        mots_recherche = list(dict.fromkeys(mots_recherche))
        resultats = []

        for plante in self.list_plante:
            texte_plante = " ".join(map(str, plante)).lower()
            mots_plante = set(
                texte_plante.replace(";", " ")
                        .replace(",", " ")
                        .split()
            )

            correspondances = sum(
                mot in mots_plante
                for mot in mots_recherche
            )

            if correspondances >= len(mots_recherche) - diff:
                resultats.append(plante)

        if not resultats:
            return "aucune correspondance avec votre recherche"

        if supprimer == False:
            return np.array([
                [f"{plante[0]} ({plante[1]}) - {plante[2]}; ", 
                 f"Cycle : {plante[3]}; ", 
                 f"Besoins : {plante[4]}"]
                for plante in resultats
            ])
        else:
            return resultats

    def nettoyer(self, Ensemble=None, silencieux=False):
        '''
        Nettoie et normalise une ligne ou une liste de lignes.
        - corrige les caractères mal encodés (mojibake), espaces, minuscules
        - supprime les accents, remplace "_" par un espace
        - ramène la famille à son nom latin (Solanacées -> Solanaceae)
        - signale (print) les valeurs et photos manquantes
        - supprime les doublons (comparaison après nettoyage)
        silencieux : si True, aucun print (utile pour les comparaisons internes)
        '''
        if Ensemble is None:
            ensemble = self.list_plante
        else:
            ensemble = Ensemble

        if isinstance(ensemble, (list, tuple, np.ndarray)) and len(ensemble) > 0 and not isinstance(ensemble[0], (list, tuple, np.ndarray)):
            ensemble = [ensemble]

        decalage = 2 if Ensemble is None else 1   # 2 = numéro de ligne dans le CSV (entête = ligne 1)
        resultat = []
        deja_vus = set()

        for i, ligne in enumerate(ensemble):
            ligne_nettoyee = []
            manquants = []
            photo_absente = False

            for idx, mot in enumerate(ligne):
                mot = str(mot).strip()

                if any(token in mot for token in ("Ã", "â", "Â", "ï¿½", "ðŸ", "�")):
                    for enc in ("latin-1", "cp1252"):
                        try:
                            mot = mot.encode(enc).decode("utf-8")
                            break
                        except Exception:
                            pass

                mot = mot.strip().replace("\ufeff", "")

                if idx != 5:                                   # tout sauf la photo
                    mot = mot.lower()
                    mot = self._sans_accent(mot)
                    mot = mot.replace("_", " ")                # arrosage_faible -> arrosage faible
                    mot = " ".join(mot.split())                # espaces multiples
                    if idx == 4:                               # besoins : "a ; b" -> "a;b"
                        mot = ";".join(b.strip() for b in mot.split(";") if b.strip())
                    if idx == 2:                               # famille
                        mot = self._famille_latine(mot)
                    if mot in self.VALEURS_VIDES:
                        mot = ""
                        manquants.append(str(self.carac[idx]))
                    if idx in (0, 1, 2):
                        mot = mot[:1].upper() + mot[1:]
                else:                                          # photo : on ne touche pas au nom du fichier
                    if mot.lower() in self.VALEURS_VIDES + (self.PHOTO_VIDE.lower(),):
                        mot = self.PHOTO_VIDE
                        photo_absente = True

                ligne_nettoyee.append(mot)

            if not silencieux:
                if manquants:
                    print(f"[valeur manquante] ligne {i + decalage} : {', '.join(manquants)}")
                if photo_absente:
                    print(f"[photo manquante] ligne {i + decalage} : '{self.PHOTO_VIDE}' utilisée")

            cle = tuple(ligne_nettoyee)
            if cle in deja_vus:
                if not silencieux:
                    print(f"[doublon] ligne {i + decalage} ignorée : {ligne_nettoyee}")
                continue
            deja_vus.add(cle)
            resultat.append(ligne_nettoyee)

        if Ensemble is None:
            self.list_plante = np.array(resultat, dtype=object)
        else:
            return resultat

    def export_herbier(self, fichier_herbier, data=None, ecraser=True):
        '''
        Exporte les données de l'herbier dans un fichier CSV.

        fichier_herbier : str
            Nom du fichier CSV.
        data : list ou np.ndarray, optionnel
            Données à enregistrer. Si None, on exporte self.list_plante.
        ecraser : bool, optionnel
            Si True, on écrase le fichier. Sinon, on ajoute les données à la fin.
        '''
        mode = 'w' if ecraser else 'a'

        if data is None:
            data = self.list_plante

        if isinstance(data, np.ndarray):
            data = data.tolist()

        if isinstance(data, list) and len(data) > 0 and not isinstance(data[0], (list, tuple, np.ndarray)):
            data = [data]

        with open(fichier_herbier, mode, newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)

            if ecraser:
                writer.writerow(['nom', 'nom_scientifique', 'famille', 'cycle', 'besoins', 'photo'])

            writer.writerows(data)

    def import_herbier(self, fichier):
        '''
        Lit un fichier CSV et le transforme en tableau NumPy.

        fichier : str
            Chemin vers le fichier CSV à charger.
        '''
        with open(fichier, "rb") as f:
            donnees = f.read()

        texte = donnees.decode("utf-8", errors="replace")

        rows = list(csv.reader(io.StringIO(texte), delimiter=",", quotechar="|"))
        return np.array(rows, dtype=object)

5. Initialiser l'objet herbier contenant les plantes sous forme de liste dans l'attribut `list_plante` 

6 et 7. Si on veut chercher précisement une plante (recherche/filtre), on utilise la méthode `rechercher`. Rechercher les plantes contenant les mots clefs demandé (à $n$ = `diff` prêt)

In [20]:
h = herbier('herbier.csv')

print("liste des plantes importées :", '\n', h.list_plante)

h.rechercher(" Urticaceae Fabaceae ", diff=1) #Pour rechercher toutes les lignes qui contiennent Solanaceae

[['nom' 'nom_scientifique' 'famille' 'cycle' 'besoins' 'photo']
 ['Tomate' 'Solanum lycopersicum' 'Solanaceae' 'annuel'
  'soleil;arrosage regulier' 'tomate.jpg']
 ['Tournesol' 'Helianthus annuus' 'Asteraceae' 'annuel' 'soleil'
  'tournesol.jpg']
 ['Haricot' 'Phaseolus vulgaris' 'Fabaceae' 'annuel'
  'soleil;arrosage moyen' 'haricot.jpg']
 ['Pois' 'Pisum sativum' 'Fabaceae' 'annuel' 'soleil;arrosage moyen'
  'pois.jpg']
 ['Lavande' 'Lavandula angustifolia' 'Lamiaceae' 'vivace'
  'soleil;arrosage faible' 'lavande.jpg']
 ['Menthe' 'Mentha sp.' 'Lamiaceae' 'vivace' 'mi-ombre;arrosage frequent'
  'aucune_photo.png']
 ['Chene' 'Quercus robur' 'Fagaceae' 'vivace' 'soleil' 'chene.jpg']
 ['Rose' 'Rosa gallica' 'Rosaceae' 'vivace' 'soleil;arrosage moyen'
  'rose.jpg']
 ['Pissenlit' 'Taraxacum officinale' 'Asteraceae' 'vivace' 'soleil'
  'pissenlit.jpg']
 ['' 'Ocimum basilicum' 'Lamiaceae' 'annuel' 'soleil;arrosage frequent'
  'basilic.jpg']
 ['Romarin' 'Salvia rosmarinus' 'Lamiaceae' 'vivace'
 

array([['Haricot (Phaseolus vulgaris) - Fabaceae; ', 'Cycle : annuel; ',
        'Besoins : soleil;arrosage moyen'],
       ['Pois (Pisum sativum) - Fabaceae; ', 'Cycle : annuel; ',
        'Besoins : soleil;arrosage moyen'],
       ['Ortie (Urtica dioica) - Urticaceae; ', 'Cycle : vivace; ',
        'Besoins : mi-ombre']], dtype='<U41')

9. Fonction nettoyer :   
Permet de normaliser les données et de supprimer les potentiels erreur d'encodage

In [12]:
h.nettoyer()           ### nettoyer toutes les données
print(h.list_plante)

[doublon] ligne 3 ignorée : ['Tomate', 'Solanum lycopersicum', 'Solanaceae', 'annuel', 'soleil;arrosage regulier', 'tomate.jpg']
[photo manquante] ligne 8 : 'aucune_photo.png' utilisée
[valeur manquante] ligne 12 : nom
[doublon] ligne 15 ignorée : ['Carotte', 'Daucus carota', 'Apiaceae', 'bisannuel', 'soleil;arrosage moyen', 'carotte.jpg']
[['Tomate' 'Solanum lycopersicum' 'Solanaceae' 'annuel'
  'soleil;arrosage regulier' 'tomate.jpg']
 ['Tournesol' 'Helianthus annuus' 'Asteraceae' 'annuel' 'soleil'
  'tournesol.jpg']
 ['Haricot' 'Phaseolus vulgaris' 'Fabaceae' 'annuel'
  'soleil;arrosage moyen' 'haricot.jpg']
 ['Pois' 'Pisum sativum' 'Fabaceae' 'annuel' 'soleil;arrosage moyen'
  'pois.jpg']
 ['Lavande' 'Lavandula angustifolia' 'Lamiaceae' 'vivace'
  'soleil;arrosage faible' 'lavande.jpg']
 ['Menthe' 'Mentha sp.' 'Lamiaceae' 'vivace' 'mi-ombre;arrosage frequent'
  'aucune_photo.png']
 ['Chene' 'Quercus robur' 'Fagaceae' 'vivace' 'soleil' 'chene.jpg']
 ['Rose' 'Rosa gallica' 'Rosaceae'

In [ ]:
['Tomate','Solanum lycopersicum','Solanacées','annuel','soleil;arrosage régulier','tomate.jpg']Test = 
print(Test)
Test_clean = h.nettoyer(Test) ### Ou uniquement une ligne avant de l'ajouter par exemple
print(Test_clean)

['Tomate', 'Solanum lycopersicum', 'Solanacées', 'annuel', 'soleil;arrosage régulier', 'tomate.jpg']
[['Tomate', 'Solanum lycopersicum', 'Solanaceae', 'annuel', 'soleil;arrosage regulier', 'tomate.jpg']]


13. Sauvegarder dans le fichier csv (il existe aussi la fonction `import_herbier`)

In [14]:
h.export_herbier('herbier.csv')

Ajouter une plante à notre liste (automatiquement sauvegardé dans le fichier csv)

In [23]:
h.ajouter_plante(Test)
print('\n')

Test2 = [
    "Capucine",
    "Tropaeolum majus",
    "Tropaeolaceae",
    "annuel",
    "soleil;arrosage modere",
    "capucine.jpg"
]

print("avant :",h.list_plante[-2:,:])
h.ajouter_plante(Test2)
print('\n')
print("après :",h.list_plante[-3:,:])

[doublon] ['Tomate', 'Solanum lycopersicum', 'Solanaceae', 'annuel', 'soleil;arrosage regulier', 'tomate.jpg'] existe déjà, non ajoutée


avant : [['Ortie' 'Urtica dioica' 'Urticaceae' 'vivace' 'mi-ombre' 'ortie.jpg']
 ['Basilic' 'Ocimum basilicum' 'Lamiaceae' 'annuel'
  'soleil;arrosage frequent' 'basilic.jpg']]


après : [['Ortie' 'Urtica dioica' 'Urticaceae' 'vivace' 'mi-ombre' 'ortie.jpg']
 ['Basilic' 'Ocimum basilicum' 'Lamiaceae' 'annuel'
  'soleil;arrosage frequent' 'basilic.jpg']
 ['Capucine' 'Tropaeolum majus' 'Tropaeolaceae' 'annuel'
  'soleil;arrosage modere' 'capucine.jpg']]


Supprimer des plantes (Pas de sauvegarde automatique, a faire a la main pour eviter de sauvegarder des erreurs)

In [22]:
h.supprimer_plante('capucine',nbr=1)

"1 lignes supprimées : [array(['Capucine', 'Tropaeolum majus', 'Tropaeolaceae', 'annuel',\n       'soleil;arrosage modere', 'capucine.jpg'], dtype=object)] supprimé, il reste 0 valide pour ce tag"

14. Menu de l'app
15. Quiz automatique

In [ ]:
## Convertir le json en csv / Inutile maintenant
import json
import csv

def json_vers_csv(fichier_json, fichier_csv):
    """Convertit un fichier JSON contenant une liste de plantes en fichier CSV."""
    with open(fichier_json, "r", encoding="utf-8") as fichier:
        donnees = json.load(fichier)

    if isinstance(donnees, dict):
        donnees = donnees.get("plantes", [donnees])

    if not donnees:
        raise ValueError("Le fichier JSON est vide.")

    champs = list(donnees[0].keys())

    with open(fichier_csv, "w", newline="", encoding="utf-8") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=champs)
        writer.writeheader()
        writer.writerows(donnees)

# Exemple
json_vers_csv("herbier.json", "herbier_json.csv")